# OnKith privacy evaluation: TinyBERT baseline and Model V2 OOD suite

This notebook is a readable orchestration layer over the tested `privacy_eval` modules. It freezes the historical IID baseline, audits the deterministic OOD partitions, and loads or runs model-only and production-pipeline comparisons. It never trains a model.

In [ ]:
from collections import Counter
import json
from pathlib import Path

from privacy_eval.schema import load_cases

ROOT = Path.cwd().resolve()
FIXTURES = ROOT / 'fixtures'
BASELINES = ROOT / 'baselines'
RESULTS = ROOT / 'results'

dev_cases = load_cases(FIXTURES / 'ood_dev.jsonl')
final_cases = load_cases(FIXTURES / 'ood_final.jsonl')
len(dev_cases), len(final_cases)

## 1. Audit the two OOD partitions

Development may guide error analysis. Final is frozen and must not guide training, augmentation, thresholds, or model selection. The manifests contain exact file hashes and generation seeds.

In [ ]:
for partition, cases in [('dev', dev_cases), ('final', final_cases)]:
    manifest = json.loads((FIXTURES / f'ood_{partition}.manifest.json').read_text(encoding='utf-8'))
    print(partition, Counter(case.category for case in cases))
    print('frozen:', manifest['frozen'], 'sha256:', manifest['fixture_sha256'])
    print('speech-like:', sum('speech_like' in case.tags for case in cases))
    print('hard negatives:', sum(case.is_hard_negative for case in cases))

## 2. Inspect controlled examples

Name groups contain title-case, lowercase, uppercase, reordered, initial, and conversational variants. This measures capitalization instead of assuming it matters to the uncased TinyBERT tokenizer. Email and numeric groups similarly pair written, reformatted, spoken, and ASR-like forms.

In [ ]:
sample_pair = next(case.pair_id for case in dev_cases if case.category == 'names')
for case in sorted((item for item in dev_cases if item.pair_id == sample_pair), key=lambda item: item.subcategory):
    print(case.subcategory, '=>', case.text)
    print('   ', [(entity.entity_type, entity.text) for entity in case.entities])

## 3. Preserve the historical IID continuity baseline

These metrics come from the existing final-test artifacts and are not recomputed here. OOD results must be reported separately because the distributions are intentionally different.

In [ ]:
iid = json.loads((BASELINES / 'tinybert_iid_baseline.json').read_text(encoding='utf-8'))
for name in ('pytorch_fp32', 'onnx_fp32', 'onnx_int8'):
    row = iid[name]
    print(name, 'token F1=', row.get('pii_token_micro_f1'), 'entity F1=', row['typed_entity_f1'], 'leakage=', row['private_character_leakage_rate'])

## 4. Run or load the TinyBERT OOD baseline

The five stages keep neural generalization, deterministic recognition, and combined production behavior separate. Set `RUN_MODELS` only when the local ONNX/Transformers environment is ready. Development-host latency is not Raspberry Pi latency.

In [ ]:
RUN_MODELS = False
if RUN_MODELS:
    from privacy_eval.run import run_partition
    run_partition(FIXTURES, RESULTS, 'dev', ('fp32_model', 'int8_model', 'structured', 'structured_supplemental', 'production'), retain_predictions=False)
else:
    print('Using existing result files; set RUN_MODELS=True to recompute the development baseline.')

In [ ]:
summary_path = RESULTS / 'tinybert_dev_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    for stage, result in summary['stages'].items():
        overall = result['overall']
        print(stage, 'typed F1=', round(overall['typed_entity']['f1'], 4), 'char recall=', round(overall['privacy_character']['private_character_recall'], 4), 'hard-negative FP=', round(overall['case_level']['hard_negative_false_positive_case_rate'], 4))
else:
    print('No full development result yet. Run the preceding cell or the CLI command in README.md.')

## 5. Frozen-final rule

Run `ood_final.jsonl` only for declared checkpoint comparisons. Never inspect final failures and then alter training data, augmentation families, thresholds, decoding, or model selection. New test ideas belong in a future versioned final partition rather than silently changing this one.